# Preprocessing

La seguente esercitazione è finalizzata a comprendere quali sono le librerie e i comandi di base più utili per eseguire pre-processing su un dataset.

Il dataset oggetto d'esame riporta l'unione di quattro studi clinici su pazienti con sindromi coronariche. Ogni record è classificato sulla base di quattro etichette che identificano diverse gravità della sindrome.

## Esercizio 0 : Importazione dei pacchetti e del dataset

In [298]:
import numpy as np
import pandas as pd # manipolazione dati e importazione(!!!)

In [299]:
# importazione dati
# NB: index_col = 0 dice che la prima colonna del dataset va usata come id per le osservazioni e non come feature
dd = pd.read_csv('data/heart_disease_uci.csv', sep = ';', index_col = 0)

In [300]:
# visualizzazione del dataset
dd

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
id,,,,,,,,,,,,,,,
1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,54,Female,VA Long Beach,asymptomatic,127.0,333.0,True,st-t abnormality,154.0,False,0.0,NaN,NaN,NaN,1
917,62,Male,VA Long Beach,typical angina,NaN,139.0,False,st-t abnormality,NaN,NaN,NaN,NaN,NaN,NaN,0
918,55,Male,VA Long Beach,asymptomatic,122.0,223.0,True,st-t abnormality,100.0,False,0.0,NaN,NaN,fixed defect,2


In [301]:
# dimensioni dataset
dd.shape

(920, 15)

## Esercizio 1: Analisi delle colonne/righe eccessivamente sparse per la rimozione

Questo esercizio a primo impatto potrebbe richiedere poco sforzo. Basta calcolare il numero di NA per colonna/riga usando la funzione `X.isna().sum(axis = 0/1)`. In realtà un'attenta analisi dovrebbe portarci a valutare quale criterio applicare per decidere se una colonna/riga deve essere **eliminata** dal dataset. Possibili criteri possono essere:
- Elimina una colonna *se ha almeno il 50% di valori mancanti*;
- Elimina una riga *se ha più del 70% di valori mancanti*.

In questa fase del pre-processing non ci soffermeremo sull'**imputazione dei dati mancanti**, che affronteremo in un secondo momento.

In [302]:
# Numero di dati mancanti per colonna
dd.isna().sum() # questo comando restituisce il numero di valori mancanti per ogni colonna del dataset

age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [303]:
# nota, axis = 0 indica che stiamo applicando la funzione colonna per colonna
dd.apply(lambda x: x.isna().sum(), axis = 0) # stesso comando, ma con apply, più lento!

age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [304]:
# quali righe hanno più valori mancanti (Top 5)
dd.isna().sum(axis = 1)

id
1      0
2      0
3      0
4      0
5      0
      ..
916    3
917    7
918    2
919    7
920    3
Length: 920, dtype: int64

Con questi comandi abbiamo ottenuto un quadro abbastanza intuitivo della situazione: le feature `ca` e `thal` hanno il numero più elevato di valori mancanti. Mentre le righe 917 e 919 hanno 7 colonne in corrispondenza delle quali mancano dei valori. 
In realtà, un quadro più completo può essere ottenuto se riusciamo a ricavare le seguenti informazioni:
1. Tra le colonne, quali hanno la percentuale più alta di dati mancanti?
2. Tra le righe, quali (e quante) hanno la percentuale più alta di dati mancanti? 

### Rimozione colonne

In [305]:
# percentuale di valori mancanti per colonna (arrotondato a 2 decimali)
dd.isna().mean().round(4) * 100 

age          0.00
sex          0.00
dataset      0.00
cp           0.00
trestbps     6.41
chol         3.26
fbs          9.78
restecg      0.22
thalch       5.98
exang        5.98
oldpeak      6.74
slope       33.59
ca          66.41
thal        52.83
num          0.00
dtype: float64

In [306]:
# quali variabili hanno più del 50% di valori mancanti? Restituisci solo nomi colonne che soddisfano la condizione
dd.columns[dd.isna().mean() > 0.5] # ca e thal 

Index(['ca', 'thal'], dtype='object')

Le variabili `ca` e `thal` superano la soglia del 50% che avevamo imposto. Se la consegna fosse quella di rimuovere tali colonne, esse andrebbero rimosse. 

**NB: la rimozione di variabili è un'operazione che può alterare i dati originali. Sarebbe sempre meglio evitare che il dato grezzo venga manipolato, per cui è opportuno**

In [307]:
# rimozione delle colonne con più del 50% di valori mancanti in un nuovo dataframe
dd_clean = dd.drop(columns = dd.columns[dd.isna().mean() > 0.5])
dd_clean.shape # le colonne ora sono 13!

(920, 13)

### Rimozione righe

Per semplicità lavoreremo sul dataframe ripulito; il numero di missing per riga può essere diverso rispetto a prima se si considera che sono state rimosse due colonne. 

In [308]:
dd_clean.isna().mean(axis = 1).round(4)* 100

id
1       0.00
2       0.00
3       0.00
4       0.00
5       0.00
       ...  
916     7.69
917    38.46
918     7.69
919    38.46
920     7.69
Length: 920, dtype: float64

Questa visualizzazione continua ad essere poco informativa. Quali sono le osservazioni (righe) con il più alto numero di NA?

In [309]:
# quali righe hanno più valori mancanti? Top 5
dd_clean.isna().mean(axis = 1).round(4).sort_values(ascending = False).head() * 100

id
885    46.15
876    46.15
902    46.15
789    38.46
889    38.46
dtype: float64

Abbiamo finalmente introdotto il metodo `sort_values()`, in cui abbiamo specificato di NON volere l'ordine crescente; dopo aver calcolato le percentuali di NA per riga, le abbiamo ordinate.

In questo caso **nessuna riga supera la soglia critica imposta**. Ma se avessimo imposto tale soglia al 40%, allora avremmo dovuto rimuovere le unità 885, 876, 902. 

In [310]:
# rimozione righe con più del 40% di valori mancanti
dd_clean = dd_clean[dd_clean.isna().mean(axis = 1) <= 0.4]
dd_clean.shape # abbiamo 3 righe in meno!

(917, 13)

Il codice scritto sino ad ora non è pulito: spesso abbiamo ripetuto la stessa sintassi continuamente. Quale può essere una soluzione ottimale? 

Sfruttiamo la **programmazione funzionale**!

In [311]:
def dropsparse(df, max_na_col = 0.5, max_na_row = 0.5):
    '''
    Rimuove le colonne e le righe con più dati mancanti rispetto alla soglia impostata. 
    df: dataframe da pulire
    max_na_col: soglia per la rimozione delle colonne (default 0.5)
    max_na_row: soglia per la rimozione delle righe (default 0.5)
    
    Restituisce un dataframe pulito.
    '''
    # 1. Rimozione colonne
    df_clean = df.drop(columns = df.columns[df.isna().mean() > max_na_col])
    # 2. Rimozione righe
    df_clean = df_clean[df_clean.isna().mean(axis = 1) <= max_na_row]
    
    return df_clean



In [312]:
dd_clean_new = dropsparse(dd, max_na_col = 0.5, max_na_row = 0.4)
dd_clean_new.shape 

(917, 13)

## Esercizio 2: Encoding delle feature categoriche

L'encoding delle feature categoriche è una fase obbligatoria per task di machine learning. È essenzialmente il processo di trasformazione di etichette testuali (es. "Rosso", "Blu", "Giallo" o "Piccolo", "Medium", "Grande") in valori numerici.

I modelli di Machine Learning svolgono compiti come il *calcolo di distanze* e l'*ottimizzazione di pesi* che richiedono l'uso di soli valori numerici.

L'encoding di feature categoriche può essere suddiviso in 3:

- **Dati nominali** (no ordinamento): quando le modalità sono distinte e non seguono un ordine logico (città, colori, marche di auto) allora bisogna usare tecniche che le trattino come tali. Si usa il **One-Hot Encoder**, che genera un vettore binario sparso;

- **Dati ordinali** (ordinamento): quando le modalità seguono un ordine ben preciso e si vuole preservare quella gerarchia. Si usa l'**Ordinal Encoder**, che assegna numeri crescenti in base all'ordine delle feature;

In [313]:
# Come prima cosa dobbiamo capire se i dati sono nominali o ordinali
# per farlo possiamo usare il metodo .unique() per vedere i valori unici di ogni colonna
dd_clean.apply(lambda x: x.unique())

age         [63, 67, 37, 41, 56, 62, 57, 53, 44, 52, 48, 5...
sex                                            [Male, Female]
dataset      [Cleveland, Hungary, Switzerland, VA Long Beach]
cp          [typical angina, asymptomatic, non-anginal, at...
trestbps    [145.0, 160.0, 120.0, 130.0, 140.0, 172.0, 150...
chol        [233.0, 286.0, 229.0, 250.0, 204.0, 236.0, 268...
fbs                                        [True, False, nan]
restecg       [lv hypertrophy, normal, st-t abnormality, nan]
thalch      [150.0, 108.0, 129.0, 187.0, 172.0, 178.0, 160...
exang                                      [False, True, nan]
oldpeak     [2.3, 1.5, 2.6, 3.5, 1.4, 0.8, 3.6, 0.6, 3.1, ...
slope                     [downsloping, flat, upsloping, nan]
num                                           [0, 2, 1, 3, 4]
dtype: object

Alcune delle variabili categoriali sono contraddistinte da `nan`, dati mancanti. Lo strumento che useremo per l'encoding **non accetta dati mancanti**, per cui bisognerà gestire questi valori mancanti *prima* di eseguire l'encoder:

1. Rimozione delle righe con missing (rischio di perdere molta informazione)

2. Imputazione semplice (es. "Ignoto", o con valore più frequente)

In ogni caso, per svolgere l'imputazione prima e l'encoding poi, bisogna introdurre il modulo `sklearn.preprocessing` di `scikit-learn`.

In [314]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

# Nativamente, sklearn restituisce array numpy, ma possiamo configurarlo per restituire DataFrame
from sklearn._config import set_config 
set_config(transform_output= 'pandas') # per far sì che le trasformazioni restituiscano DataFrame

In [315]:
# Crea due Imputer distinti per dati categorici e numerici
imp_cat = SimpleImputer(strategy = 'most_frequent') # oppure strategy = 'constant', fill_value = 'missing'
imp_num = SimpleImputer(strategy = 'median') # oppure strategy = 'mean'

# Identifica colonne numeriche e categoriche
num_cols = dd_clean.select_dtypes(include = ['number']).columns
cat_cols = dd_clean.select_dtypes(exclude = ['number']).columns

# Imputazione dati mancanti
dd_imp = dd_clean.copy()

dd_imp[cat_cols] = imp_cat.fit_transform(dd_clean[cat_cols])
dd_imp[num_cols] = imp_num.fit_transform(dd_clean[num_cols])

dd_imp.isna().sum()


age         0
sex         0
dataset     0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
num         0
dtype: int64

In [316]:
dd_imp.dtypes 

age         float64
sex          object
dataset      object
cp           object
trestbps    float64
chol        float64
fbs          object
restecg      object
thalch      float64
exang        object
oldpeak     float64
slope        object
num         float64
dtype: object

Dobbiamo assegnare la strategia corretta alle colonne a cui vogliamo applicare l'encoder. Il tutto può essere automatizzato, ma è necessario che vengano specificate **quali variabili categoriali sono anche ordinali e qual è l'ordine delle modalità** che l'ordinal encoder deve rispettare.

In [317]:
# Nel nostro caso slope sembra essere l'unica variabile ordinale
ord_cols = ['slope']
slope_order = [['missing', 'downsloping', 'flat', 'upsloping']] # da meno a più grave

# Filtra le colonne nominali
nom_cols = [col for col in cat_cols if col not in ord_cols]

Invece di eseguire l'encoding colonna per colonna, Scikit-Learn consente di usare un tool chiamato `ColumnTransformer`, dal modulo `sklearn.compose`, che consente di applicare encoders diversi a gruppi di colonne in un unico passaggio.

In [318]:
from sklearn.compose import ColumnTransformer

In [319]:
preprocessor = ColumnTransformer(
    transformers=[
       
        ('nominal', OneHotEncoder(drop='if_binary', sparse_output=False), nom_cols),
        
        ('ordinal', OrdinalEncoder(categories=slope_order), ord_cols),
        
        ('numerical', 'passthrough', num_cols)
    ],
    remainder='passthrough' 
)
dd_enc = preprocessor.fit_transform(dd_imp)

In [320]:
# tipo colonne
dd_enc.apply(lambda x: x.unique())

nominal__sex_Male                                                           [1.0, 0.0]
nominal__dataset_Cleveland                                                  [1.0, 0.0]
nominal__dataset_Hungary                                                    [0.0, 1.0]
nominal__dataset_Switzerland                                                [0.0, 1.0]
nominal__dataset_VA Long Beach                                              [0.0, 1.0]
nominal__cp_asymptomatic                                                    [0.0, 1.0]
nominal__cp_atypical angina                                                 [0.0, 1.0]
nominal__cp_non-anginal                                                     [0.0, 1.0]
nominal__cp_typical angina                                                  [1.0, 0.0]
nominal__fbs_True                                                           [1.0, 0.0]
nominal__restecg_lv hypertrophy                                             [1.0, 0.0]
nominal__restecg_normal                    

L'encoding è stato eseguito, adesso possiamo procedere.

## Esercizio 3. Split train/test

Una fase importante per l'adattamento di modelli di ML, qualsiasi essi siano, è lo splitting del dataset in `train` e `test`, solitamente usando proporzioni 90%-10% (ma può essere richiesto diversamente). Lo split del dataset va fatto **a monte**, poiché tutte le fasi dell'analisi devono essere ben separate per evitare **data leakage**, cioé che i dati di addestramento influenzino quelli di test.

In [321]:
# Split train/ test
from sklearn.model_selection import train_test_split

Per correttezza, lo split sarà eseguito sul primo dataset (originale) caricato. 

In [322]:
## NB: se il dataset è sbilanciato, è importante usare stratify = y per mantenere la stessa proporzione di classi in train e test

# Separa le feature di addestramento dalla variabile target
X = dd.drop(columns = ['num'])
y = dd['num']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size = 0.1, stratify = y, random_state = 42)

In termini metodologici, ciò che dovrebbe essere fatto è  **(1) splittare, poi (2) separatamente eseguire le trasformazioni/imputazioni sui dati** e infine eseguire l'analisi di multicollinearità. In particolare immaginiamo le fasi di un ipotetico task di pre-processing che richieda:

0. Analisi esplorativa (dati mancanti, describe) -> Rimuovi righe o colonne(?)
1. Splitting 90-10 (già eseguito);
2. Imputazione dei dati mancanti;
3. Encoding delle variabili categoriali;
4. Standardizzazione;
5. Studio della correlazione;
6. Feature selection

In [323]:
dd = pd.read_csv('data/heart_disease_uci.csv', sep = ';', index_col = 0)

dd.describe().round(2)


,age,trestbps,chol,thalch,oldpeak,ca,num
count,920.00,861.00,890.00,865.00,858.00,309.00,920.00
mean,53.51,132.13,199.13,137.55,0.88,0.68,1.00
std,9.42,19.07,110.78,25.93,1.09,0.94,1.14
min,28.00,0.00,0.00,60.00,-2.60,0.00,0.00
25%,47.00,120.00,175.00,120.00,0.00,0.00,0.00
50%,54.00,130.00,223.00,140.00,0.50,0.00,1.00
75%,60.00,140.00,268.00,157.00,1.50,1.00,2.00
max,77.00,200.00,603.00,202.00,6.20,3.00,4.00


In [324]:
dd.isna().mean().round(4)

age         0.0000
sex         0.0000
dataset     0.0000
cp          0.0000
trestbps    0.0641
chol        0.0326
fbs         0.0978
restecg     0.0022
thalch      0.0598
exang       0.0598
oldpeak     0.0674
slope       0.3359
ca          0.6641
thal        0.5283
num         0.0000
dtype: float64

Il target `num` non presenta dati mancanti. `ca` e `thal` hanno +50% di missing; fissiamo la soglia massima di NA accettabili per riga al 40%.

In [325]:
def dropsparse(df, max_na_col = 0.5, max_na_row = 0.5):
    df_clean = df.drop(columns = df.columns[df.isna().mean() >= max_na_col])
    df_clean = df_clean[df_clean.isna().mean(axis = 1) <= max_na_row]
    return df_clean

dd_clean = dropsparse(dd, max_na_col = 0.5, max_na_row = 0.4)
dd_clean.shape


(917, 13)

Imputazione per dati mancanti con la moda (categoriali) e mediana (numeriche)

In [326]:
dd_clean.apply(lambda x: x.unique())

age         [63, 67, 37, 41, 56, 62, 57, 53, 44, 52, 48, 5...
sex                                            [Male, Female]
dataset      [Cleveland, Hungary, Switzerland, VA Long Beach]
cp          [typical angina, asymptomatic, non-anginal, at...
trestbps    [145.0, 160.0, 120.0, 130.0, 140.0, 172.0, 150...
chol        [233.0, 286.0, 229.0, 250.0, 204.0, 236.0, 268...
fbs                                        [True, False, nan]
restecg       [lv hypertrophy, normal, st-t abnormality, nan]
thalch      [150.0, 108.0, 129.0, 187.0, 172.0, 178.0, 160...
exang                                      [False, True, nan]
oldpeak     [2.3, 1.5, 2.6, 3.5, 1.4, 0.8, 3.6, 0.6, 3.1, ...
slope                     [downsloping, flat, upsloping, nan]
num                                           [0, 2, 1, 3, 4]
dtype: object

In [327]:
imp_cat = SimpleImputer(strategy = 'most_frequent')
imp_num = SimpleImputer(strategy = 'median')

In [328]:
X = dd_clean.drop(columns = ['num'])
y = dd_clean['num']

num_cols = X.select_dtypes(include = ['number']).columns
cat_cols = X.select_dtypes(exclude = ['number']).columns

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size = 0.1, stratify = y, random_state= 44)

# L'imputazione va eseguita distintamente tra train e test set
X_tr[cat_cols] = imp_cat.fit_transform(X_tr[cat_cols])
X_te[cat_cols] = imp_cat.transform(X_te[cat_cols])

X_tr[num_cols] = imp_num.fit_transform(X_tr[num_cols])
X_te[num_cols] = imp_num.transform(X_te[num_cols])

Ordinal Encoding per variabili categoriali (uno per le nominali, uno per le ordinali). 

**NB: .fit_transform sul training set, .transform sul test set**.

In [329]:
ord_cols = ['slope']
ord_cats = [['downsloping', 'flat', 'upsloping']]

cat_cols = [col for col in cat_cols if col != 'slope']

enc1 = OrdinalEncoder()
enc2 = OrdinalEncoder(categories= ord_cats)

X_tr[cat_cols] = enc1.fit_transform(X_tr[cat_cols])
X_tr[ord_cols] = enc2.fit_transform(X_tr[ord_cols])

X_te[cat_cols] = enc1.transform(X_te[cat_cols])
X_te[ord_cols] = enc2.transform(X_te[ord_cols])

Adesso avviene la fase in cui va eseguita la **standardizzazione**.

In [330]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

# media e std = (1, 0)
X_tr_s.describe() 

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope
count,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02,8.250000e+02
mean,-1.679465e-16,3.229740e-17,5.167584e-17,1.464149e-16,-7.751375e-17,-9.581561e-17,-4.952268e-17,6.890111e-17,-9.473903e-17,9.689219e-17,4.521636e-17,-2.002439e-16
std,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00,1.000607e+00
min,-2.731591e+00,-1.941336e+00,-1.096737e+00,-8.298858e-01,-7.148491e+00,-1.813500e+00,-4.205831e-01,-1.584213e+00,-3.110624e+00,-7.678511e-01,-2.686262e+00,-2.199805e+00
25%,-7.017395e-01,5.151091e-01,-1.096737e+00,-8.298858e-01,-6.343701e-01,-2.001337e-01,-4.205831e-01,1.160596e-02,-7.129553e-01,-7.678511e-01,-8.072793e-01,-2.894480e-01
50%,4.610062e-02,5.151091e-01,-2.087997e-01,-8.298858e-01,-9.152669e-02,2.191593e-01,-4.205831e-01,1.160596e-02,8.626763e-02,-7.678511e-01,-3.375336e-01,-2.894480e-01
75%,6.871065e-01,5.151091e-01,6.791372e-01,1.283249e+00,4.513167e-01,6.202221e-01,-4.205831e-01,1.160596e-02,7.256460e-01,1.302336e+00,6.019578e-01,-2.894480e-01
max,2.503290e+00,5.151091e-01,1.567074e+00,2.339817e+00,3.708377e+00,3.682884e+00,2.377651e+00,1.607425e+00,2.284131e+00,1.302336e+00,5.017567e+00,1.620909e+00


In [331]:
X_te_s.describe()

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope
count,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000
mean,-0.078152,-0.045601,-0.025421,-0.025976,0.171044,0.079659,-0.055596,-0.040432,-0.062284,-0.070288,-0.045474,0.022023
std,1.065672,1.036648,1.015719,1.100579,1.019141,0.950296,0.947556,1.069883,1.061601,0.983877,0.952457,0.862614
min,-2.624757,-1.941336,-1.096737,-0.829886,-1.720057,-1.813500,-0.420583,-1.584213,-2.711012,-0.767851,-3.249957,-2.199805
25%,-0.915408,0.515109,-1.096737,-0.829886,-0.634370,-0.140886,-0.420583,0.011606,-0.842829,-0.767851,-0.807279,-0.289448
50%,-0.060734,0.515109,-0.208800,-0.829886,-0.091527,0.223717,-0.420583,0.011606,0.086268,-0.767851,-0.337534,-0.289448
75%,0.580272,0.515109,0.679137,1.283249,0.722738,0.627058,-0.420583,0.011606,0.705665,1.302336,0.601958,-0.289448
max,2.396455,0.515109,1.567074,2.339817,3.708377,3.008369,2.377651,1.607425,2.563859,1.302336,2.762788,1.620909


## Esercizio 4: Imputazione delle feature mancanti

Questa fase di pre-processing, successiva allo splitting prevede in linea di massima l'uso di due strategie di SimpleImputer, `median` per le quantitative e `most_frequent` per le categoriali. 

## Esercizio 5: Analisi della correlazione

L'analisi di correlazione serve a rimuovere le variabili per eliminare multicollinearità tra le feature esplicative.

L'analisi di correlazione può essere svolta in due fasi:

1. Correlazione bassa con la risposta, in questo caso le esplicative sono poco utili per eseguire il task;

2. Correlazione alta tra le esplicative, problemi di multicollinearità e ridondanza.

In [332]:
soglia = 0.1

cl_to_remove = (
    X_tr_s.corrwith(y_tr)
    .abs()
    .sort_values(ascending = False)
    .loc[lambda x: x < soglia]
    .index
)

cl_to_remove

Index(['restecg'], dtype='object')

`restecg` appare poco correlata alla risposta. Rimuoviamola dal dataset, poi calcoliamo la correlazione tra le esplicative del training set rimanenti.

In [333]:
# Train set filtrato 
X_tr_flt = X_tr_s.drop(columns = cl_to_remove)
X_tr_flt.shape

(825, 11)

In [334]:
soglia_mat = 0.8

coppie = (
    X_tr_flt
    .corr()
    .abs()
    .map(lambda x: x > soglia_mat)
    .unstack()
    .reset_index()
)

coppie.columns = ['var1', 'var2', 'high_corr']

( coppie
    .loc[
        (coppie['var1'] < coppie['var2']) & # per eliminare duplicati e non ripetere coppie
        (True == coppie['high_corr']),
        ['var1', 'var2']
    ]
)

,var1,var2


Nessuna coppia mostra una correlazione al di sopra di 0.8; non rimuoviamo ulteriori coppie.

In [335]:
#Rimuovine una dalle coppie altamente correlate
new_cl_to_remove = [] # Inserisci nomi

tot_cl_to_remove = cl_to_remove.union(new_cl_to_remove)
tot_cl_to_remove

Index(['restecg'], dtype='object')

In [336]:
X_tr_flt = X_tr_s.drop(columns = tot_cl_to_remove)
X_te_flt = X_te_s.drop(columns = tot_cl_to_remove)

In [337]:
X_tr_flt.isna().mean()

age         0.0
sex         0.0
dataset     0.0
cp          0.0
trestbps    0.0
chol        0.0
fbs         0.0
thalch      0.0
exang       0.0
oldpeak     0.0
slope       0.0
dtype: float64

## Esercizio 5: Feature Selection

La selezione delle feature rilevanti, dopo una prima scrematura successiva all'analisi di correlazione, è necessaria a **ridurre la dimensionalità mantenendo l'informazione**. 

Per eseguire questo task ci viene in aiuto il modulo `sklearn.feature_selection` e in particolare le funzioni `SelectKBest` e `SelectFromModel`. Vediamo come funzionano nel dettaglio:

- `SelectKBest`: esegue un filtraggio delle variabili basato sui risultati di test statistici come l'ANOVA F-test (*per variabili continue*) o il test del Chi-Quadro (*per variabili categoriali, o se la y è categoriale*). Generalmente, per compiti di classificazione con feature miste tra quantitative e encodizzate, `f_classif` è il metodo di scoring preferito. 

- `SelectFromModel`: un "problema" della tecnica precedente è il fatto che non valuta le interazioni tra le feature rispetto alla variabile target. Questa tecnica cattura le interazioni e risolve anche la scelta manuale di un numero *K* di feature da selezionare, in quanto **imposta una soglia automatica** basata sull'importanza delle feature. 

In [338]:
from sklearn.feature_selection import SelectKBest, f_classif

# Top-5 feature per Anova F-test, che indica la forza della relazione tra ogni feature e la variabile target 
skb = SelectKBest(score_func=f_classif, k = 5) 

# Fit SOLO su train set, Transform solo su test per evitare data leakage
X_tr_skb = skb.fit_transform(X_tr_flt, y_tr)
X_te_skb = skb.transform(X_te_flt)

X_tr_skb.shape # 5 feature selezionate

(825, 5)

In [339]:
X_tr_skb.columns.to_list() # nomi delle feature selezionate

['age', 'cp', 'thalch', 'exang', 'oldpeak']

La tecnica ha selezionato le 5 feature più correlate alla variabile target, senza tenere conto della relazione tra le feature stesse.

### Come funziona `SelectFromModel`

Un'importante distinzione nell'uso di questa funzione dipende dal tipo di variabile risposta che riguarda il problema considerato.

1. Se la risposta è **discreta/categoriale** (es. predire `0` vs `1`, `vivo` vs `morto` o, come in questo caso, per problemi multiclasse): si usano **Modelli di Classificazione**;
2. Se la risposta è  **continua/numerica** (es. predire la pressione sanguigna, prezzi di case, etc.): si usano **Modelli di regressione**.

Nel nostro caso `num` è una variabile multiclasse con etichette numeriche; se la trattassimo come variabile quantitativa continua usando un modello di regressione, allora introdurremmo una forzatura interpretativa, immaginando che la distanza tra lo stadio 1 e 2 sia la stessa tra lo stadio 2 e 3.

A loro volta i modelli di classificazione/regressione possono essere suddivisi in tre principali gruppi. 

- Modelli basati sul modulo `linear_model`;

- Modelli basati sui moduli `svm`, `tree`, `neighbors`;

- Modelli basati sul modulo `ensemble`.

Poiché rientramo nel caso multi-classe, dobbiamo ricadere sui metodi di classificazione. Immaginiamo di voler usare una selezione delle feature basata sul modello di Random Forest.

In [341]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Impostando threshold = 'median', si selezionano le feature con importanza maggiore di quella mediana
sfm = SelectFromModel(RandomForestClassifier(n_estimators= 100, random_state= 42), threshold = 'median')
X_tr_sfm = sfm.fit_transform(X_tr_flt, y_tr)
X_te_sfm = sfm.transform(X_te_flt)

X_tr_sfm.columns.to_list() # nomi delle feature selezionate

['age', 'cp', 'trestbps', 'chol', 'thalch', 'oldpeak']

4 delle feature selezionate con `SelectKBest` sono state selezionate anche dal modello Random Forest. 

### Sintassi dei principali metodi di Feature Selection con `SelectFromModel` (CLASSIFICAZIONE)

1. Regressione logistica (linear_model, classificatore)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

# penalty='l1' permette di ridurre a zero i coefficienti delle feature meno importanti
# solver='liblinear' è un algoritmo di ottimizzazione che supporta la regolarizzazione L1
sfm = SelectFromModel(
    LogisticRegression(penalty='l1', solver='liblinear', random_state=42),
    threshold='1e-5' # soglia molto bassa per selezionare tutte le feature con coefficiente diverso da zero
)

X_tr = sfm.fit_transform(X_tr, y_tr)
X_te = sfm.transform(X_te)

2. Support Vector Classifier (svm, embedded)

In [ ]:
from sklearn.svm import LinearSVC

# Usa i coefficienti del modello con regolarizzazione L1 per selezionare le feature più importanti
sfm = SelectFromModel(
    LinearSVC(penalty='l1', dual=False, random_state=42), # dual = False è necessario quando si usa la regolarizzazione L1
    threshold='median'
)

X_tr_sfm = sfm.fit_transform(X_tr, y_tr)
X_te_sfm = sfm.transform(X_te)

3. Decision Tree (tree, embedded) e metodi ensemble (ensemble, embedded)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier

# Sostituisci i modelli inseriti in SelectFromModel:
# - DecisionTreeClassifier(random_state=42)
# - RandomForestClassifier(n_estimators=100, random_state=42)
# - GradientBoostingClassifier(random_state=42)
# - AdaBoostClassifier(random_state=42)
# - HistGradientBoostingClassifier(random_state=42)

sfm = SelectFromModel(
    GradientBoostingClassifier(random_state=42), 
    threshold='median'
)
X_tr_sfm = sfm.fit_transform(X_tr, y_tr)
X_te_sfm = sfm.transform(X_te)

### Sintassi dei principali metodi di Feature Selection con `SelectFromModel` (REGRESSIONE)

1. Metodo basato su regressione (Lasso) con grid search del parametro `alpha` 

**NB**: il valore di $\alpha$ ha un impatto sulla forza della penalità applicata ai coefficienti del modello. Un $\alpha$ troppo basso (prossimo a 0) implica una penalità bassa, facendo performare il modello come una regressione lineare standard, mentre un $\alpha$ troppo alto implica una penalizzazione troppo aggressiva che porta *tutti i coefficienti a zero*, rendendo il modello inutile.


In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFromModel

# 1. Definisci una griglia di valori per il parametro alpha da testare
# La best practice è testare valori su scala logaritmica, ad esempio da 0.0001 a 10
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10]}

# 2. Imposta la Grid Search usando la cross validation (es. 5-fold CV)
# Si usa 'r2' come metrica di valutazione, che è standard per i modelli di regressione
# n_jobs=-1 permette di utilizzare tutti i core del processore per velocizzare la ricerca
grid_search = GridSearchCV(
    estimator=Lasso(max_iter=10000, random_state=42), 
    param_grid=param_grid, 
    cv=5, 
    scoring='r2', # per classificazione si usa scoring='accuracy' o 'f1'
    n_jobs=-1
)

# 3. Adatta la grid search al training set
grid_search.fit(X_tr, y_tr)
best_alpha = grid_search.best_params_['alpha']

# 4. Inserisci il modello ottimizzato in SelectFromModel
sfm = SelectFromModel(
    grid_search.best_estimator_, 
    threshold='1e-5'
)

# 5. Applica la selezione delle feature al training e test set
X_tr_sfm = sfm.fit_transform(X_tr_flt, y_tr)
X_te_sfm = sfm.transform(X_te_flt)

X_tr_sfm.columns.tolist() # nomi delle feature selezionate


Questo metodo, sebbene funzioni, può diventare oneroso computazionalmente al crescere della dimensione della griglia e del numero di osservazioni. Una versione che ha un costo computazionale inferiore è quella offerta dalla funzione `LassoCV` dello stesso modulo:

In [ ]:
from sklearn.linear_model import LassoCV

# LassoCV esegue automaticamente la cross validation per trovare il miglior alpha
lasso_cv = LassoCV(alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10], cv = 5, random_state=42, max_iter=10000)
lasso_cv.fit(X_tr, y_tr)
best_alpha = lasso_cv.alpha 

sfm = SelectFromModel(
    lasso_cv,
    threshold = '1e-5'
)

X_tr_sfm = sfm.fit_transform(X_tr, y_tr)
X_te_sfm = sfm.transform(X_te)

2. Support Vector Regression (svm, embedded)

In [ ]:
from sklearn.svm import SVR

# Funziona solo se il kernel è impostato su 'linear'
sfm = SelectFromModel(
    SVR(kernel='linear'), 
    threshold='median'
)

X_tr_sfm = sfm.fit_transform(X_tr, y_tr)
X_te_sfm = sfm.transform(X_te)

3. Alberi di regressione e metodi ensemble (tree/ensemble, embedded)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

# Sostituisci i modelli inseriti in SelectFromModel:
# - DecisionTreeRegressor(random_state=42)
# - RandomForestRegressor(random_state=42)
# - GradientBoostingRegressor(random_state=42)
# - HistGradientBoostingRegressor(random_state=42)

sfm = SelectFromModel(
    RandomForestRegressor(random_state=42), 
    threshold='median'
)

X_tr_sfm = sfm.fit_transform(X_tr, y_tr)
X_te_sfm = sfm.transform(X_te)

# Riepilogo 

## 0. Importazione e visualizzazione dati

Abbiamo imparato che per questa fase è indispensabile lavorare con `pandas` per l'importazione dei dati con `pd.read_csv('percorso/dati.csv')`, specificando eventualmente il separatore se non è la virgola di default, e se leggere la prima colonna come un id.
- `X.head()` per visualizzare le prime righe e colonne del dataframe;
- `X.shape()` per visualizzare il numero di (righe, colonne).

## 1. Analisi dei dati mancanti (riga e colonna)

- `X.isna().mean(axis = 0/1)` per vedere la proporzione di NA per colonna/riga;

- Aggiungendo il metodo `.sort_values(ascending = False)` possiamo ordinarle per numero di missing

- La rimozione delle colonne sfrutta la sintassi `X.drop(columns = X.columns[X.isna().mean >= 0.5])`;

- La rimozione delle righe sfrutta la sintassi `X[X.isna().mean(axis = 1) <= 0.5]`, tecnicamente in questo caso stiamo *conservando le righe con al più tot% NA*.